In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

In [3]:
w=torch.tensor(3.0) # 0차원 스칼라
b=torch.tensor(-1.0)
print(w,b)

tensor(3.) tensor(-1.)


In [6]:
def forward(x):
    # what=3X-1
    yhat=w*x+b
    return yhat

In [10]:
x=torch.tensor(1.0)
y_foward=forward(x)
print(y_foward)
print(y_foward.item())

tensor(2.)
2.0


In [9]:
# torch.Tensor(3) # 크기가 3인 1차원 텐서
# torch.tensor(3) # 값이 3인 0차원 텐서

In [13]:
x=torch.tensor([[1.0,3.0,5.0]])
yhat=forward(x)
print(x.size()) # 입력사이즈
print(yhat.shape) # 출력사이즈

torch.Size([1, 3])
torch.Size([1, 3])


In [29]:
from torch.nn import Linear
torch.manual_seed(1)
# 입력데이터 1개, 예측값 1개, 편향 사용할거임(기본이 True)
model=Linear(in_features=1, out_features=1, bias=True)
# 가중치, 편향(무작위 값)
print(model.state_dict())
print(model.state_dict()['weight'])
print(model.state_dict()['bias'])
print(model.weight)
print(model.bias)

OrderedDict([('weight', tensor([[0.5153]])), ('bias', tensor([-0.4414]))])
tensor([[0.5153]])
tensor([-0.4414])
Parameter containing:
tensor([[0.5153]], requires_grad=True)
Parameter containing:
tensor([-0.4414], requires_grad=True)


In [30]:
# 커스텀 클래스(사용자 정의)
from torch import nn
class LinearReg(nn.Module):
    def __init__(self, in_size, out_size):
        super().__init__() # 부모클래스 nn.Module의 초기화 메소드 호출(모델 등록하고-> 가중치 추적가능해짐)
        self.linear=nn.Linear(in_size, out_size) # y=wx+b

    # 우리가 줄 x(모델에게 줄 입력데이터)
    # w,b(nn.Linear 내부적으로 랜덤하게 줌)
    # out 계산된다
    def forward(self, x):
        out=self.linear(x) # self.linear.__call(x)
        return out

In [34]:
model=LinearReg(1,1)
model.state_dict()

OrderedDict([('linear.weight', tensor([[-0.2057]])),
             ('linear.bias', tensor([0.5087]))])

In [36]:
x=torch.tensor([[1.1],[2.1],[3.1]]) # 입력값
yhat=model(x) # LinearReg 객체인 model을 함수로 호출을 해서 파이토치 내부의 __call__
print(yhat)

tensor([[ 0.2824],
        [ 0.0767],
        [-0.1290]], grad_fn=<AddmmBackward0>)


In [33]:
class A:
    def __call__(self,x):
         return x+1

a=A()
print(a(3))

4


In [37]:
#########################################################

In [45]:
# requires_grad : 미분 대상으로 설정
x=torch.rand(1, requires_grad=True)
y=torch.rand(1, requires_grad=True)
loss=y-x
loss
# loss를 y로 편미분 => 1
# loss를 x로 편미분 => -1

tensor([-0.0086], grad_fn=<SubBackward0>)

In [46]:
loss.backward()
print(x.grad, y.grad)

tensor([-1.]) tensor([1.])


In [47]:
#########################################################

In [58]:
x=torch.zeros(4) #입력
y=torch.ones(3)
w=torch.rand(4,3,requires_grad=True) # 가중치
b=torch.rand(3, requires_grad=True) # 편향
# y2=wx+b => b: 입력에 상관없이 모델이 가지는 값
y2=torch.matmul(x,w)+b # (4),(4,3) => (3,)+(3,)= >(3,)
print(b)
print(y2) # 예측값 grad_fn=<AddBackward0>) -> backward 할 때 더하기 연산 역추적하겠다. -> 기록해둠

tensor([0.7165, 0.1768, 0.0748], requires_grad=True)
tensor([0.7165, 0.1768, 0.0748], grad_fn=<AddBackward0>)


In [59]:
import torch.nn.functional as F
loss=F.mse_loss(y2,y) # (y2-y)의 제곱 후 평균낸다
loss.backward()
print(w.grad, b.grad, loss)
# loss가 있는데 grad 0 => 데이터가 0이기 때문에
# loss가 너무 크고 grad 도 차이가 많이 남 다> 학습률 (learning rate)가 너무 높을 수 있음
# loss, grad 적절 => 파라미터 업데이트 시킴

tensor([[-0., -0., -0.],
        [-0., -0., -0.],
        [-0., -0., -0.],
        [-0., -0., -0.]]) tensor([-0.1890, -0.5488, -0.6168]) tensor(0.5380, grad_fn=<MseLossBackward0>)


In [ ]:
# 1. forward 함수를 예측값 내기
# 2. loss로 실제값-예측값
# 3. backward 로 grad 계산 (역순)
# MSE - 덧셈미분 - 행렬곱
# 4. optimizer 로 계산 w,b 값 update